<a href="https://colab.research.google.com/github/Nosheen-Akhtar16/intelligent-inventory-demand-system/blob/main/Inventory_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

print("Environment ready!")

Environment ready!


In [ ]:
df = pd.read_csv("retail_store_inventory.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(73100, 15)
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [ ]:
print("Shape:", df.shape)
print("\nData Types:\n")
print(df.dtypes)

print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nUnique Values:\n")
print(df.nunique())

for col in df.columns:
    print(f"\n{col}")
    print(df[col].unique()[:10])

Shape: (73100, 15)

Data Types:

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition      object
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality            object
dtype: object

Missing Values:

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forecast       0
Price                 0
Discount              0
Weather Condition     0
Holiday/Promotion     0
Competitor Pricing    0
Seasonality           0
dtype: int64

Unique Values:

Date                    731
Store ID                  5
Product ID               20
Category   

In [ ]:
import re

def normalize_column_name(column):
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = column.strip("_")
    return column

normalized_columns = {
    col: normalize_column_name(col)
    for col in df.columns
}

normalized_columns

{'Date': 'date',
 'Store ID': 'store_id',
 'Product ID': 'product_id',
 'Category': 'category',
 'Region': 'region',
 'Inventory Level': 'inventory_level',
 'Units Sold': 'units_sold',
 'Units Ordered': 'units_ordered',
 'Demand Forecast': 'demand_forecast',
 'Price': 'price',
 'Discount': 'discount',
 'Weather Condition': 'weather_condition',
 'Holiday/Promotion': 'holiday_promotion',
 'Competitor Pricing': 'competitor_pricing',
 'Seasonality': 'seasonality'}

In [ ]:
COLUMN_ALIASES = {
    "date": [
        "date", "datetime", "timestamp", "transaction_date",
        "order_date", "sales_date"
    ],
    "sales": [
        "sales", "units_sold", "quantity_sold",
        "sales_quantity", "sales_qty", "units_sales"
    ],
    "inventory": [
        "inventory", "inventory_level", "stock",
        "stock_level", "available_stock", "current_stock"
    ],
    "price": [
        "price", "unit_price", "selling_price", "product_price"
    ],
    "discount": [
        "discount", "discount_rate", "discount_percent"
    ],
    "promotion": [
        "promotion", "promotions", "holiday_promotion",
        "promo", "is_promotion"
    ],
    "store_id": [
        "store_id", "store", "store_code"
    ],
    "product_id": [
        "product_id", "product", "sku", "sku_id", "item_id"
    ],
    "category": [
        "category", "product_category", "item_category"
    ],
    "region": [
        "region", "area", "location", "market"
    ],
    "competitor_price": [
        "competitor_price", "competitor_pricing",
        "competitor_cost"
    ],
    "weather": [
        "weather", "weather_condition"
    ],
    "seasonality": [
        "seasonality", "season", "season_type"
    ]
}

def detect_columns(df):
    normalized = {
        col: normalize_column_name(col)
        for col in df.columns
    }

    detected = {}

    for role, aliases in COLUMN_ALIASES.items():
        for original, normalized_name in normalized.items():
            if normalized_name in aliases:
                detected[role] = original
                break

    return detected

detected_columns = detect_columns(df)

for role, column in detected_columns.items():
    print(f"{role:20} → {column}")

date                 → Date
sales                → Units Sold
inventory            → Inventory Level
price                → Price
discount             → Discount
promotion            → Holiday/Promotion
store_id             → Store ID
product_id           → Product ID
category             → Category
region               → Region
competitor_price     → Competitor Pricing
weather              → Weather Condition
seasonality          → Seasonality


In [ ]:
CAPABILITY_REQUIREMENTS = {
    "demand_forecasting": ["date", "sales"],
    "stockout_risk": ["sales", "inventory"],
    "anomaly_detection": ["date", "sales"],
    "price_analysis": ["price", "sales"],
    "promotion_analysis": ["promotion", "sales"],
    "regional_analysis": ["region", "sales"]
}

def detect_capabilities(detected_columns):
    capabilities = {}

    available_roles = set(detected_columns.keys())

    for module, required_roles in CAPABILITY_REQUIREMENTS.items():
        missing = [
            role for role in required_roles
            if role not in available_roles
        ]

        capabilities[module] = {
            "available": len(missing) == 0,
            "missing": missing
        }

    return capabilities


capabilities = detect_capabilities(detected_columns)

for module, result in capabilities.items():
    status = "AVAILABLE" if result["available"] else "UNAVAILABLE"
    print(f"{module:25} → {status}")

    if result["missing"]:
        print(f"   Missing: {result['missing']}")

demand_forecasting        → AVAILABLE
stockout_risk             → AVAILABLE
anomaly_detection         → AVAILABLE
price_analysis            → AVAILABLE
promotion_analysis        → AVAILABLE
regional_analysis         → AVAILABLE


In [ ]:
def analyze_schema(df):
    normalized = {
        col: normalize_column_name(col)
        for col in df.columns
    }

    detected = detect_columns(df)
    capabilities = detect_capabilities(detected)

    return {
        "original_columns": list(df.columns),
        "normalized_columns": normalized,
        "detected_roles": detected,
        "capabilities": capabilities
    }


schema_report = analyze_schema(df)

print("DETECTED ROLES")
for role, column in schema_report["detected_roles"].items():
    print(f"{role:20} → {column}")

print("\nCAPABILITIES")
for module, result in schema_report["capabilities"].items():
    status = "AVAILABLE" if result["available"] else "UNAVAILABLE"
    print(f"{module:25} → {status}")

    if result["missing"]:
        print(f"   Missing: {result['missing']}")

DETECTED ROLES
date                 → Date
sales                → Units Sold
inventory            → Inventory Level
price                → Price
discount             → Discount
promotion            → Holiday/Promotion
store_id             → Store ID
product_id           → Product ID
category             → Category
region               → Region
competitor_price     → Competitor Pricing
weather              → Weather Condition
seasonality          → Seasonality

CAPABILITIES
demand_forecasting        → AVAILABLE
stockout_risk             → AVAILABLE
anomaly_detection         → AVAILABLE
price_analysis            → AVAILABLE
promotion_analysis        → AVAILABLE
regional_analysis         → AVAILABLE


In [ ]:
def validate_data(df, detected_columns):

    validation = {}

    for role, column in detected_columns.items():

        series = df[column]

        validation[role] = {
            "column": column,
            "dtype": str(series.dtype),
            "missing": int(series.isna().sum()),
            "unique_values": int(series.nunique()),
            "duplicates": int(series.duplicated().sum())
        }

    return validation


validation_report = validate_data(df, detected_columns)

for role, result in validation_report.items():
    print(f"\n{role.upper()}")
    print(f"Column: {result['column']}")
    print(f"Dtype: {result['dtype']}")
    print(f"Missing: {result['missing']}")
    print(f"Unique values: {result['unique_values']}")
    print(f"Duplicate values: {result['duplicates']}")


DATE
Column: Date
Dtype: object
Missing: 0
Unique values: 731
Duplicate values: 72369

SALES
Column: Units Sold
Dtype: int64
Missing: 0
Unique values: 498
Duplicate values: 72602

INVENTORY
Column: Inventory Level
Dtype: int64
Missing: 0
Unique values: 451
Duplicate values: 72649

PRICE
Column: Price
Dtype: float64
Missing: 0
Unique values: 8999
Duplicate values: 64101

DISCOUNT
Column: Discount
Dtype: int64
Missing: 0
Unique values: 5
Duplicate values: 73095

PROMOTION
Column: Holiday/Promotion
Dtype: int64
Missing: 0
Unique values: 2
Duplicate values: 73098

STORE_ID
Column: Store ID
Dtype: object
Missing: 0
Unique values: 5
Duplicate values: 73095

PRODUCT_ID
Column: Product ID
Dtype: object
Missing: 0
Unique values: 20
Duplicate values: 73080

CATEGORY
Column: Category
Dtype: object
Missing: 0
Unique values: 5
Duplicate values: 73095

REGION
Column: Region
Dtype: object
Missing: 0
Unique values: 4
Duplicate values: 73096

COMPETITOR_PRICE
Column: Competitor Pricing
Dtype: float64


In [ ]:
print(" EXACT DUPLICATE ROWS ")
print("Duplicate rows:", df.duplicated().sum())

print("\n DATE CHECK ")
date_col = detected_columns["date"]

parsed_dates = pd.to_datetime(df[date_col], errors="coerce")

print("Invalid dates:", parsed_dates.isna().sum())
print("Start date:", parsed_dates.min())
print("End date:", parsed_dates.max())
print("Unique dates:", parsed_dates.nunique())

print("\n NUMERIC VALUE CHECK")

for role in ["sales", "inventory", "price", "discount"]:
    col = detected_columns[role]

    print(f"\n{role.upper()} ({col})")
    print("Min:", df[col].min())
    print("Max:", df[col].max())
    print("Negative values:", (df[col] < 0).sum())

 EXACT DUPLICATE ROWS 
Duplicate rows: 0

 DATE CHECK 
Invalid dates: 0
Start date: 2022-01-01 00:00:00
End date: 2024-01-01 00:00:00
Unique dates: 731

 NUMERIC VALUE CHECK

SALES (Units Sold)
Min: 0
Max: 499
Negative values: 0

INVENTORY (Inventory Level)
Min: 50
Max: 500
Negative values: 0

PRICE (Price)
Min: 10.0
Max: 100.0
Negative values: 0

DISCOUNT (Discount)
Min: 0
Max: 20
Negative values: 0


In [ ]:
date_col = detected_columns["date"]
store_col = detected_columns["store_id"]
product_col = detected_columns["product_id"]

df_check = df.copy()
df_check[date_col] = pd.to_datetime(df_check[date_col])

group_sizes = (
    df_check
    .groupby([store_col, product_col])[date_col]
    .agg(["min", "max", "nunique"])
)

print(" STORE-PRODUCT HISTORY ")
print("Number of series:", len(group_sizes))
print("Minimum observations:", group_sizes["nunique"].min())
print("Maximum observations:", group_sizes["nunique"].max())

expected_days = (df_check[date_col].max() - df_check[date_col].min()).days + 1

print("\nExpected days:", expected_days)
print("Complete series:", (group_sizes["nunique"] == expected_days).sum())
print("Incomplete series:", (group_sizes["nunique"] < expected_days).sum())

 STORE-PRODUCT HISTORY 
Number of series: 100
Minimum observations: 731
Maximum observations: 731

Expected days: 731
Complete series: 100
Incomplete series: 0


In [ ]:
#Data Repair Layer
def repair_data_types(df, detected_columns):

    repaired_df = df.copy()
    repair_log = []

    for role, column in detected_columns.items():

        if role == "date":
            original_dtype = str(repaired_df[column].dtype)
            repaired_df[column] = pd.to_datetime(
                repaired_df[column],
                errors="coerce"
            )

            if original_dtype != str(repaired_df[column].dtype):
                repair_log.append(
                    f"{column}: converted to datetime"
                )

        elif role in [
            "sales",
            "inventory",
            "price",
            "discount",
            "competitor_price"
        ]:
            original_dtype = str(repaired_df[column].dtype)

            converted = pd.to_numeric(
                repaired_df[column],
                errors="coerce"
            )

            repaired_df[column] = converted

            if original_dtype != str(converted.dtype):
                repair_log.append(
                    f"{column}: converted to numeric"
                )

    return repaired_df, repair_log


repaired_df, repair_log = repair_data_types(
    df,
    detected_columns
)

print("DATA TYPE REPAIR")

if repair_log:
    for item in repair_log:
        print("✓", item)
else:
    print("No repairs needed.")

print("\nFINAL DTYPES")
print(repaired_df[list(detected_columns.values())].dtypes)

#Missing value repair


def repair_missing_values(df, detected_columns):

    repaired_df = df.copy()
    repair_log = []

    for role, column in detected_columns.items():

        missing_before = repaired_df[column].isna().sum()

        if missing_before == 0:
            continue

        if role in [
            "sales",
            "inventory",
            "price",
            "discount",
            "competitor_price"
        ]:
            repaired_df[column] = repaired_df[column].fillna(
                repaired_df[column].median()
            )

            method = "median"

        elif role in [
            "store_id",
            "product_id",
            "category",
            "region",
            "weather",
            "seasonality"
        ]:
            repaired_df[column] = repaired_df[column].fillna(
                repaired_df[column].mode()[0]
            )

            method = "mode"

        elif role == "promotion":
            repaired_df[column] = repaired_df[column].fillna(0)
            method = "zero"

        elif role == "date":
            repaired_df[column] = repaired_df[column].ffill()

            method = "forward fill"

        else:
            continue

        missing_after = repaired_df[column].isna().sum()

        repair_log.append({
            "column": column,
            "missing_before": int(missing_before),
            "missing_after": int(missing_after),
            "method": method
        })

    return repaired_df, repair_log


repaired_df, repair_log = repair_missing_values(
    repaired_df,
    detected_columns
)

print("\n MISSING VALUE REPAIR ")

if repair_log:
    for item in repair_log:
        print(
            f"{item['column']}: "
            f"{item['missing_before']} → "
            f"{item['missing_after']} "
            f"({item['method']})"
        )
else:
    print(" No missing values found. No repairs needed.")

    #Invalid value detection
def detect_invalid_values(df, detected_columns):

    issues = {}

    checks = {
        "sales": lambda s: s < 0,
        "inventory": lambda s: s < 0,
        "price": lambda s: s < 0,
        "discount": lambda s: s < 0,
        "competitor_price": lambda s: s < 0
    }

    for role, check in checks.items():

        if role not in detected_columns:
            continue

        column = detected_columns[role]
        invalid_count = check(df[column]).sum()

        issues[role] = {
            "column": column,
            "invalid_count": int(invalid_count)
        }

    return issues


invalid_report = detect_invalid_values(
    repaired_df,
    detected_columns
)

print("\nINVALID VALUE CHECK")

for role, result in invalid_report.items():

    if result["invalid_count"] == 0:
        print(f"{result['column']}: PASS")
    else:
        print(
            f"{result['column']}: "
            f"{result['invalid_count']} invalid values"
        )

def validate_dates(df, detected_columns):

    if "date" not in detected_columns:
        return None

    column = detected_columns["date"]

    invalid_dates = df[column].isna().sum()

    return {
        "column": column,
        "invalid_dates": int(invalid_dates)
    }

date_report = validate_dates(
    repaired_df,
    detected_columns
)

print("\nDATE VALIDATION")

if date_report["invalid_dates"] == 0:
    print(f"{date_report['column']}: PASS")
else:
    print(
        f"{date_report['column']}: "
        f"{date_report['invalid_dates']} invalid dates"
    )

DATA TYPE REPAIR
✓ Date: converted to datetime

FINAL DTYPES
Date                  datetime64[ns]
Units Sold                     int64
Inventory Level                int64
Price                        float64
Discount                       int64
Holiday/Promotion              int64
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Competitor Pricing           float64
Weather Condition             object
Seasonality                   object
dtype: object

 MISSING VALUE REPAIR 
 No missing values found. No repairs needed.

INVALID VALUE CHECK
Units Sold: PASS
Inventory Level: PASS
Price: PASS
Discount: PASS
Competitor Pricing: PASS

DATE VALIDATION
Date: PASS


In [ ]:
# Demand Signal Sanity Check

sales_col = detected_columns["sales"]
inventory_col = detected_columns["inventory"]

sales_exceed_inventory = (
    repaired_df[sales_col] > repaired_df[inventory_col]
).sum()

total_rows = len(repaired_df)

percentage = (
    sales_exceed_inventory / total_rows
) * 100

print("DEMAND SIGNAL CHECK")
print(f"Total records: {total_rows}")
print(f"Sales > Inventory: {sales_exceed_inventory}")
print(f"Percentage: {percentage:.2f}%")


#inventory relattionships
print("\nSALES SUMMARY")
print(repaired_df[sales_col].describe())

print("\nINVENTORY SUMMARY")
print(repaired_df[inventory_col].describe())

#actual forcasting series
date_col = detected_columns["date"]
store_col = detected_columns["store_id"]
product_col = detected_columns["product_id"]

series_count = repaired_df.groupby(
    [store_col, product_col]
).ngroups

print(f"Number of forecasting series: {series_count}")

DEMAND SIGNAL CHECK
Total records: 73100
Sales > Inventory: 0
Percentage: 0.00%

SALES SUMMARY
count    73100.000000
mean       136.464870
std        108.919406
min          0.000000
25%         49.000000
50%        107.000000
75%        203.000000
max        499.000000
Name: Units Sold, dtype: float64

INVENTORY SUMMARY
count    73100.000000
mean       274.469877
std        129.949514
min         50.000000
25%        162.000000
50%        273.000000
75%        387.000000
max        500.000000
Name: Inventory Level, dtype: float64
Number of forecasting series: 100


In [ ]:
sample_series = (
    repaired_df[
        (repaired_df[store_col] == "S001") &
        (repaired_df[product_col] == "P0001")
    ]
    .sort_values(date_col)
)

print(sample_series[
    [date_col, sales_col, inventory_col]
].head(15))

           Date  Units Sold  Inventory Level
0    2022-01-01         127              231
100  2022-01-02          81              116
200  2022-01-03           5              154
300  2022-01-04          58               85
400  2022-01-05         147              238
500  2022-01-06          37              198
600  2022-01-07         107              195
700  2022-01-08           2              231
800  2022-01-09         350              373
900  2022-01-10          36              327
1000 2022-01-11          16              101
1100 2022-01-12         104              145
1200 2022-01-13          80              110
1300 2022-01-14         177              199
1400 2022-01-15         176              290


In [ ]:


from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

results = []

test_size = 30

grouped = repaired_df.groupby(
    [store_col, product_col]
)

for (store, product), group in grouped:

    # Sort by date
    group = group.sort_values(date_col)

    # Create proper time series with DatetimeIndex
    series = (
        group
        .set_index(date_col)[sales_col]
        .astype(float)
    )

    # Make sure frequency is daily
    series = series.asfreq("D")

    # Skip if not enough observations
    if len(series) <= test_size + 14:
        continue

    # Train-test split
    train_series = series.iloc[:-test_size]
    test_series = series.iloc[-test_size:]

    try:

        # Exponential Smoothing
        model = ExponentialSmoothing(
            train_series,
            trend="add",
            seasonal="add",
            seasonal_periods=7
        )

        fitted_model = model.fit(
            optimized=True
        )

        # Forecast
        predictions = fitted_model.forecast(
            steps=test_size
        )

        # Evaluation
        mae = mean_absolute_error(
            test_series,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                test_series,
                predictions
            )
        )

        results.append({
            "store": store,
            "product": product,
            "observations": len(series),
            "mae": mae,
            "rmse": rmse
        })

    except Exception as e:

        print(
            f"Skipped {store} - {product}: {e}"
        )


# RESULTS
results_df = pd.DataFrame(results)

print("ALL SERIES EVALUATION")
print("----------------------")

print(
    f"Series evaluated: {len(results_df)}"
)

if len(results_df) > 0:

    print(
        f"Average MAE: "
        f"{results_df['mae'].mean():.2f}"
    )

    print(
        f"Average RMSE: "
        f"{results_df['rmse'].mean():.2f}"
    )

    print(
        f"Median MAE: "
        f"{results_df['mae'].median():.2f}"
    )

    print(
        f"Median RMSE: "
        f"{results_df['rmse'].median():.2f}"
    )

    print("\nBEST 5 SERIES BY MAE")

    print(
        results_df
        .sort_values("mae")
        .head(5)
        .to_string(index=False)
    )

    print("\nWORST 5 SERIES BY MAE")

    print(
        results_df
        .sort_values("mae", ascending=False)
        .head(5)
        .to_string(index=False)
    )

else:

    print("No series could be evaluated.")

ALL SERIES EVALUATION
----------------------
Series evaluated: 100
Average MAE: 87.43
Average RMSE: 105.85
Median MAE: 87.72
Median RMSE: 104.69

BEST 5 SERIES BY MAE
store product  observations       mae      rmse
 S004   P0015           731 66.908529 74.628921
 S003   P0007           731 70.738526 89.477158
 S001   P0008           731 70.819428 83.201817
 S001   P0009           731 70.826643 85.917287
 S004   P0016           731 70.928042 89.163867

WORST 5 SERIES BY MAE
store product  observations        mae       rmse
 S002   P0007           731 125.646783 141.644166
 S005   P0002           731 116.515264 140.154027
 S003   P0018           731 107.468039 128.878982
 S002   P0008           731 105.482963 119.866336
 S005   P0006           731 103.466125 131.536024


In [ ]:

# DEMAND FORECASTING FUNCTION
# EXPONENTIAL SMOOTHING
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pandas as pd
import numpy as np


def forecast_demand(
    df,
    store,
    product,
    forecast_days=30
):

    # Select required store-product series
    series_df = df[
        (df[store_col] == store) &
        (df[product_col] == product)
    ].copy()

    # Sort chronologically
    series_df = series_df.sort_values(date_col)

    # Create daily time series
    series = (
        series_df
        .set_index(date_col)[sales_col]
        .astype(float)
        .asfreq("D")
    )

    # Check sufficient history
    if len(series) < 30:
        raise ValueError(
            "Not enough historical data for forecasting."
        )

    # Train Exponential Smoothing model
    model = ExponentialSmoothing(
        series,
        trend="add",
        seasonal="add",
        seasonal_periods=7
    )

    fitted_model = model.fit(
        optimized=True
    )

    # Generate future forecast
    forecast = fitted_model.forecast(
        steps=forecast_days
    )

    # Create forecast output
    forecast_output = pd.DataFrame({
        "date": forecast.index,
        "store": store,
        "product": product,
        "forecasted_demand": forecast.values
    })

    # Prevent negative demand predictions
    forecast_output["forecasted_demand"] = (
        forecast_output["forecasted_demand"]
        .clip(lower=0)
    )

    return forecast_output



# TEST DEMAND FORECAST

forecast_result = forecast_demand(
    repaired_df,
    store="S001",
    product="P0001",
    forecast_days=30
)

print("DEMAND FORECAST")

print(
    forecast_result.to_string(index=False)
)

DEMAND FORECAST
      date store product  forecasted_demand
2024-01-02  S001   P0001         126.522955
2024-01-03  S001   P0001         138.872814
2024-01-04  S001   P0001         144.636177
2024-01-05  S001   P0001         143.861752
2024-01-06  S001   P0001         135.883840
2024-01-07  S001   P0001         144.377738
2024-01-08  S001   P0001         121.846813
2024-01-09  S001   P0001         126.508821
2024-01-10  S001   P0001         138.858681
2024-01-11  S001   P0001         144.622043
2024-01-12  S001   P0001         143.847619
2024-01-13  S001   P0001         135.869707
2024-01-14  S001   P0001         144.363605
2024-01-15  S001   P0001         121.832679
2024-01-16  S001   P0001         126.494688
2024-01-17  S001   P0001         138.844547
2024-01-18  S001   P0001         144.607909
2024-01-19  S001   P0001         143.833485
2024-01-20  S001   P0001         135.855573
2024-01-21  S001   P0001         144.349471
2024-01-22  S001   P0001         121.818545
2024-01-23  S001

In [ ]:

# LEVEL 5 — STOCKOUT RISK
# HISTORICAL 7-DAY DEMAND
stockout_df = repaired_df.copy()

stockout_df = stockout_df.sort_values(
    [store_col, product_col, date_col]
)

# Calculate actual demand over the NEXT 7 days
stockout_df["future_7d_demand"] = (
    stockout_df
    .groupby([store_col, product_col])[sales_col]
    .transform(
        lambda x: x.shift(-1)
                  .rolling(7, min_periods=7)
                  .sum()
                  .shift(-6)
    )
)

print("STOCKOUT RISK DATA")

print(
    stockout_df[
        [
            store_col,
            product_col,
            date_col,
            inventory_col,
            sales_col,
            "future_7d_demand"
        ]
    ].head(15).to_string(index=False)
)

STOCKOUT RISK DATA
Store ID Product ID       Date  Inventory Level  Units Sold  future_7d_demand
    S001      P0001 2022-01-01              231         127             437.0
    S001      P0001 2022-01-02              116          81             706.0
    S001      P0001 2022-01-03              154           5             737.0
    S001      P0001 2022-01-04               85          58             695.0
    S001      P0001 2022-01-05              238         147             652.0
    S001      P0001 2022-01-06              198          37             695.0
    S001      P0001 2022-01-07              195         107             765.0
    S001      P0001 2022-01-08              231           2             939.0
    S001      P0001 2022-01-09              373         350             686.0
    S001      P0001 2022-01-10              327          36             719.0
    S001      P0001 2022-01-11              101          16             826.0
    S001      P0001 2022-01-12              1

In [ ]:
# ==========================================
# STOCKOUT RISK — REPLENISHMENT ANALYSIS
# ==========================================

ordered_col = "Units Ordered"

# Basic statistics
print("UNITS ORDERED")
print("----------------")

print(
    repaired_df[ordered_col].describe()
)

# Compare inventory, orders and sales
print("\nAVERAGES")
print("----------------")

print(
    f"Average Inventory: "
    f"{repaired_df[inventory_col].mean():.2f}"
)

print(
    f"Average Units Sold: "
    f"{repaired_df[sales_col].mean():.2f}"
)

print(
    f"Average Units Ordered: "
    f"{repaired_df[ordered_col].mean():.2f}"
)

# Check how often orders are placed
order_days = (
    repaired_df[ordered_col] > 0
).sum()

total_rows = len(repaired_df)

print("\nREPLENISHMENT FREQUENCY")
print("-----------------------")

print(
    f"Days with orders: {order_days}"
)

print(
    f"Percentage: "
    f"{(order_days / total_rows) * 100:.2f}%"
)

# Compare future demand with inventory + current orders
stockout_df = stockout_df.merge(
    repaired_df[
        [
            store_col,
            product_col,
            date_col,
            ordered_col
        ]
    ],
    on=[store_col, product_col, date_col],
    how="left"
)

stockout_df["available_with_orders"] = (
    stockout_df[inventory_col]
    + stockout_df[ordered_col]
)

stockout_df["coverage_with_orders"] = (
    stockout_df["available_with_orders"]
    / stockout_df["future_7d_demand"]
)

print("\nCOVERAGE WITH REPLENISHMENT")
print("---------------------------")

print(
    stockout_df[
        [
            inventory_col,
            ordered_col,
            "future_7d_demand",
            "coverage_with_orders"
        ]
    ].head(15).to_string(index=False)
)

UNITS ORDERED
----------------
count    73100.000000
mean       110.004473
std         52.277448
min         20.000000
25%         65.000000
50%        110.000000
75%        155.000000
max        200.000000
Name: Units Ordered, dtype: float64

AVERAGES
----------------
Average Inventory: 274.47
Average Units Sold: 136.46
Average Units Ordered: 110.00

REPLENISHMENT FREQUENCY
-----------------------
Days with orders: 73100
Percentage: 100.00%

COVERAGE WITH REPLENISHMENT
---------------------------
 Inventory Level  Units Ordered  future_7d_demand  coverage_with_orders
             231             55             437.0              0.654462
             116            104             706.0              0.311615
             154            189             737.0              0.465400
              85            193             695.0              0.400000
             238             37             652.0              0.421779
             198            155             695.0              0.

In [ ]:
# ==========================================
# COVERAGE RATIO DISTRIBUTION
# ==========================================

coverage = (
    stockout_df["inventory_coverage_ratio"]
)

print("INVENTORY COVERAGE RATIO")
print("------------------------")

print(
    coverage.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

print("\nCOVERAGE PERCENTILES")
print("--------------------")

for p in [10, 25, 50, 75, 90]:
    print(
        f"{p}th percentile: "
        f"{coverage.quantile(p / 100):.3f}"
    )

INVENTORY COVERAGE RATIO
------------------------
count    72400.000000
mean         0.318285
std          0.197927
min          0.025562
10%          0.099558
25%          0.169216
50%          0.287586
75%          0.422429
90%          0.565485
max          3.543860
Name: inventory_coverage_ratio, dtype: float64

COVERAGE PERCENTILES
--------------------
10th percentile: 0.100
25th percentile: 0.169
50th percentile: 0.288
75th percentile: 0.422
90th percentile: 0.565


In [ ]:
# ==========================================
# CHECK POSSIBLE STOCKOUT RISK THRESHOLDS
# ==========================================

coverage = stockout_df["inventory_coverage_ratio"]

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("POSSIBLE RISK THRESHOLDS")
print("------------------------")

for threshold in thresholds:

    at_risk = (coverage < threshold).sum()
    safe = (coverage >= threshold).sum()

    at_risk_pct = (at_risk / len(coverage)) * 100
    safe_pct = (safe / len(coverage)) * 100

    print(
        f"Threshold {threshold:.2f} → "
        f"AT_RISK: {at_risk_pct:.2f}% | "
        f"SAFE: {safe_pct:.2f}%"
    )

POSSIBLE RISK THRESHOLDS
------------------------
Threshold 0.20 → AT_RISK: 31.48% | SAFE: 68.52%
Threshold 0.25 → AT_RISK: 42.05% | SAFE: 57.95%
Threshold 0.30 → AT_RISK: 52.52% | SAFE: 47.48%
Threshold 0.35 → AT_RISK: 62.62% | SAFE: 37.38%
Threshold 0.40 → AT_RISK: 71.44% | SAFE: 28.56%
Threshold 0.45 → AT_RISK: 78.85% | SAFE: 21.15%
Threshold 0.50 → AT_RISK: 84.54% | SAFE: 15.46%


In [ ]:
# ==========================================
# FINAL STOCKOUT RISK LABEL
# ==========================================

RISK_THRESHOLD = 0.30

stockout_df["stockout_risk"] = (
    stockout_df["inventory_coverage_ratio"]
    < RISK_THRESHOLD
).astype(int)

# 1 = AT_RISK
# 0 = SAFE

print("FINAL STOCKOUT RISK LABEL")
print("-------------------------")

print(
    stockout_df["stockout_risk"]
    .value_counts()
    .sort_index()
)

print("\nPERCENTAGE")
print("-------------------------")

print(
    stockout_df["stockout_risk"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nLABEL MEANING")
print("-------------------------")
print("0 = SAFE")
print("1 = AT_RISK")

FINAL STOCKOUT RISK LABEL
-------------------------
stockout_risk
0    34376
1    38024
Name: count, dtype: int64

PERCENTAGE
-------------------------
stockout_risk
0    47.48
1    52.52
Name: proportion, dtype: float64

LABEL MEANING
-------------------------
0 = SAFE
1 = AT_RISK


In [ ]:
# ==========================================
# STOCKOUT RISK FEATURE ENGINEERING
# ==========================================

risk_df = stockout_df.copy()

risk_df = risk_df.sort_values(
    [store_col, product_col, date_col]
).reset_index(drop=True)

group_cols = [store_col, product_col]

# Previous-day sales
risk_df["sales_lag_1"] = (
    risk_df.groupby(group_cols)[sales_col]
    .shift(1)
)

# Previous 7-day sales average
risk_df["sales_rolling_7"] = (
    risk_df.groupby(group_cols)[sales_col]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

# Previous 14-day sales average
risk_df["sales_rolling_14"] = (
    risk_df.groupby(group_cols)[sales_col]
    .transform(
        lambda x: x.shift(1).rolling(14).mean()
    )
)

# Recent sales trend
risk_df["sales_trend"] = (
    risk_df["sales_rolling_7"]
    - risk_df["sales_rolling_14"]
)

# Inventory relative to recent demand
risk_df["inventory_to_recent_sales"] = (
    risk_df[inventory_col]
    / (risk_df["sales_rolling_7"] + 1)
)

print("FEATURES CREATED")
print("----------------")

print(
    risk_df[
        [
            inventory_col,
            "sales_lag_1",
            "sales_rolling_7",
            "sales_rolling_14",
            "sales_trend",
            "inventory_to_recent_sales"
        ]
    ].head(10)
)
# ==========================================
# ADD BUSINESS FEATURES
# ==========================================

feature_columns = [
    inventory_col,
    ordered_col,
    "sales_lag_1",
    "sales_rolling_7",
    "sales_rolling_14",
    "sales_trend",
    "inventory_to_recent_sales",
    "Price",
    "Discount",
    "Holiday/Promotion",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

print("FEATURE COLUMNS")
print("----------------")

for feature in feature_columns:
    print(feature)
# ==========================================
# REMOVE INITIAL ROWS WITHOUT HISTORY
# ==========================================

risk_df = risk_df.dropna(
    subset=[
        "sales_lag_1",
        "sales_rolling_7",
        "sales_rolling_14",
        "sales_trend",
        "inventory_to_recent_sales"
    ]
).copy()

print("Rows after feature engineering:", len(risk_df))

print("\nRisk distribution:")
print(
    risk_df["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

FEATURES CREATED
----------------
   Inventory Level  sales_lag_1  sales_rolling_7  sales_rolling_14  \
0              231          NaN              NaN               NaN   
1              116        127.0              NaN               NaN   
2              154         81.0              NaN               NaN   
3               85          5.0              NaN               NaN   
4              238         58.0              NaN               NaN   
5              198        147.0              NaN               NaN   
6              195         37.0              NaN               NaN   
7              231        107.0        80.285714               NaN   
8              373          2.0        62.428571               NaN   
9              327        350.0       100.857143               NaN   

   sales_trend  inventory_to_recent_sales  
0          NaN                        NaN  
1          NaN                        NaN  
2          NaN                        NaN  
3          NaN     

In [ ]:
# ==========================================
# TIME-AWARE TRAIN / TEST SPLIT
# ==========================================

risk_df = risk_df.sort_values(
    [store_col, product_col, date_col]
).reset_index(drop=True)

test_days = 30

test_indices = (
    risk_df
    .groupby([store_col, product_col])
    .tail(test_days)
    .index
)

test_df = risk_df.loc[test_indices].copy()
train_df = risk_df.drop(test_indices).copy()

print("TRAIN / TEST SPLIT")
print("------------------")

print("Training rows:", len(train_df))
print("Testing rows :", len(test_df))

print("\nTraining date range:")
print(train_df[date_col].min(), "→", train_df[date_col].max())

print("\nTesting date range:")
print(test_df[date_col].min(), "→", test_df[date_col].max())
print("TRAIN TARGET DISTRIBUTION")

print(
    train_df["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTEST TARGET DISTRIBUTION")
print("------------------------")

print(
    test_df["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAIN / TEST SPLIT
------------------
Training rows: 68000
Testing rows : 3000

Training date range:
2022-01-15 00:00:00 → 2023-11-25 00:00:00

Testing date range:
2023-11-26 00:00:00 → 2023-12-25 00:00:00
TRAIN TARGET DISTRIBUTION
------------------------
stockout_risk
1    52.52
0    47.48
Name: proportion, dtype: float64

TEST TARGET DISTRIBUTION
------------------------
stockout_risk
1    52.5
0    47.5
Name: proportion, dtype: float64


In [ ]:
# ==========================================
# PREPARE FEATURES
# ==========================================

numeric_features = [
    inventory_col,
    ordered_col,
    "sales_lag_1",
    "sales_rolling_7",
    "sales_rolling_14",
    "sales_trend",
    "inventory_to_recent_sales",
    "Price",
    "Discount",
    "Holiday/Promotion"
]

categorical_features = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

target = "stockout_risk"

X_train = train_df[
    numeric_features + categorical_features
].copy()

X_test = test_df[
    numeric_features + categorical_features
].copy()

y_train = train_df[target]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression trained successfully.")

X_train: (68000, 14)
X_test : (3000, 14)
y_train: (68000,)
y_test : (3000,)
Processed training shape: (68000, 27)
Processed testing shape : (3000, 27)
Logistic Regression trained successfully.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred = logistic_model.predict(X_test_processed)

print("LOGISTIC REGRESSION RESULTS")
print("===========================")

print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))

print("\nClassification Report")
print("---------------------")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["SAFE", "AT_RISK"]
    )
)

print("\nConfusion Matrix")
print("----------------")

print(confusion_matrix(y_test, y_pred))

# FINAL STOCKOUT RISK MODEL

final_risk_model = logistic_model

print("Final stockout risk model: Logistic Regression")

LOGISTIC REGRESSION RESULTS
Accuracy : 0.8507
Precision: 0.8507
Recall   : 0.8679
F1 Score : 0.8592

Classification Report
---------------------
              precision    recall  f1-score   support

        SAFE       0.85      0.83      0.84      1425
     AT_RISK       0.85      0.87      0.86      1575

    accuracy                           0.85      3000
   macro avg       0.85      0.85      0.85      3000
weighted avg       0.85      0.85      0.85      3000


Confusion Matrix
----------------
[[1185  240]
 [ 208 1367]]
Final stockout risk model: Logistic Regression


In [ ]:
# ==========================================
# TEST DECISION ENGINE - CORRECT VERSION
# ==========================================

store = "S001"
product = "P0001"

# ------------------------------------------
# 1. Generate 7-day demand forecast
# ------------------------------------------

forecast_result = forecast_demand(
    repaired_df,
    store,
    product,
    forecast_days=7
)

forecasted_demand = (
    forecast_result["forecasted_demand"].sum()
)


# ------------------------------------------
# 2. Get latest row WITH engineered features
# ------------------------------------------

latest_row = (
    risk_df[
        (risk_df[store_col] == store) &
        (risk_df[product_col] == product)
    ]
    .sort_values(date_col)
    .iloc[-1]
)

current_inventory = latest_row[inventory_col]


# ------------------------------------------
# 3. Prepare classifier input
# ------------------------------------------

latest_features = pd.DataFrame(
    [latest_row]
)

latest_features = latest_features[
    numeric_features + categorical_features
]


# ------------------------------------------
# 4. Preprocess features
# ------------------------------------------

latest_processed = preprocessor.transform(
    latest_features
)


# ------------------------------------------
# 5. Predict stockout risk
# ------------------------------------------

risk_probability = logistic_model.predict_proba(
    latest_processed
)[0][1]


# ------------------------------------------
# 6. Generate business decision
# ------------------------------------------

decision = generate_decision(
    current_inventory,
    forecasted_demand,
    risk_probability
)


# ------------------------------------------
# 7. Generate explanation
# ------------------------------------------

explanation = explain_decision(
    current_inventory,
    forecasted_demand,
    risk_probability,
    decision
)


# ------------------------------------------
# 8. Final output
# ------------------------------------------

print("DECISION ENGINE OUTPUT")
print("======================")

print("Store:", store)
print("Product:", product)

print(
    "Current inventory:",
    current_inventory
)

print(
    "7-day forecasted demand:",
    round(forecasted_demand, 2)
)

print(
    "Stockout risk probability:",
    round(risk_probability, 4)
)

print(
    "Decision:",
    decision
)

print(
    "Explanation:",
    explanation
)

DECISION ENGINE OUTPUT
Store: S001
Product: P0001
Current inventory: 109
7-day forecasted demand: 956.0
Stockout risk probability: 0.9745
Decision: URGENT REORDER
Explanation: Current inventory is 109 units against forecasted demand of 956 units. Inventory coverage is 0.11x and predicted stockout risk is 97.5%. Recommended action: URGENT REORDER.


In [ ]:
# INTEGRATED INVENTORY ANALYSIS

def analyze_inventory(
    store,
    product,
    historical_df=repaired_df,
    feature_df=risk_df
):

    # 1. Generate 7-day demand forecast
    forecast_result = forecast_demand(
        historical_df,
        store,
        product,
        forecast_days=7
    )

    forecasted_demand = (
        forecast_result["forecasted_demand"].sum()
    )

    # 2. Get latest engineered row

    series_df = feature_df[
        (feature_df[store_col] == store) &
        (feature_df[product_col] == product)
    ].sort_values(date_col)

    if series_df.empty:
        raise ValueError(
            f"No data found for store={store}, product={product}"
        )

    latest_row = series_df.iloc[-1]

    current_inventory = latest_row[inventory_col]

    # 3. Prepare risk-model features
    latest_features = pd.DataFrame(
        [latest_row]
    )

    latest_features = latest_features[
        numeric_features + categorical_features
    ]

    latest_processed = preprocessor.transform(
        latest_features
    )

    # 5. Predict stockout risk
    risk_probability = (
        final_risk_model
        .predict_proba(latest_processed)[0][1]
    )

    risk_label = (
        "AT_RISK"
        if risk_probability >= 0.50
        else "SAFE"
    )


    # --------------------------------------
    # 6. Generate decision
    # --------------------------------------

    decision = generate_decision(
        current_inventory,
        forecasted_demand,
        risk_probability
    )


    # --------------------------------------
    # 7. Generate explanation
    # --------------------------------------

    explanation = explain_decision(
        current_inventory,
        forecasted_demand,
        risk_probability,
        decision
    )


    # --------------------------------------
    # 8. Calculate coverage
    # --------------------------------------

    coverage_ratio = (
        current_inventory / forecasted_demand
        if forecasted_demand > 0
        else None
    )


    # 9. Return structured result

    return {
        "store": store,
        "product": product,
        "current_inventory": float(current_inventory),
        "forecasted_demand_7d": float(
            forecasted_demand
        ),
        "inventory_coverage_ratio": (
            round(coverage_ratio, 4)
            if coverage_ratio is not None
            else None
        ),
        "stockout_risk_probability": round(
            float(risk_probability),
            4
        ),
        "stockout_risk": risk_label,
        "decision": decision,
        "explanation": explanation
    }


result = analyze_inventory(
    store="S001",
    product="P0001"
)

print("INTEGRATED ANALYSIS")

for key, value in result.items():
    print(f"{key}: {value}")

INTEGRATED ANALYSIS
store: S001
product: P0001
current_inventory: 109.0
forecasted_demand_7d: 956.0020901373434
inventory_coverage_ratio: 0.114
stockout_risk_probability: 0.9745
stockout_risk: AT_RISK
decision: URGENT REORDER
explanation: Current inventory is 109 units against forecasted demand of 956 units. Inventory coverage is 0.11x and predicted stockout risk is 97.5%. Recommended action: URGENT REORDER.


In [ ]:

# LEVEL 6 — ANOMALY DETECTION
anomaly_df = risk_df.copy()

anomaly_features = [
    sales_col,
    inventory_col,
    ordered_col,
    "sales_lag_1",
    "sales_rolling_7",
    "sales_rolling_14",
    "sales_trend",
    "inventory_to_recent_sales",
    "Price",
    "Discount",
    "Holiday/Promotion"
]

print("ANOMALY FEATURES")
print("----------------")

for feature in anomaly_features:
    print(feature)

# PREPARE ANOMALY DATA

anomaly_data = anomaly_df[
    anomaly_features
].copy()

anomaly_data = anomaly_data.replace(
    [np.inf, -np.inf],
    np.nan
)

anomaly_data = anomaly_data.dropna()

print("Rows:", len(anomaly_data))
print("Features:", anomaly_data.shape[1])
from sklearn.ensemble import IsolationForest

# ISOLATION FOREST
anomaly_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

anomaly_model.fit(
    anomaly_data
)

print("Isolation Forest trained successfully.")

# GENERATE ANOMALY SCORES
anomaly_df["anomaly_prediction"] = (
    anomaly_model.predict(
        anomaly_df[anomaly_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
)

anomaly_df["anomaly_score"] = (
    anomaly_model.decision_function(
        anomaly_df[anomaly_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
)

# Isolation Forest:
#  1  = normal
# -1  = anomaly

anomaly_df["is_anomaly"] = (
    anomaly_df["anomaly_prediction"] == -1
).astype(int)

print("ANOMALY RESULTS")

print(
    anomaly_df["is_anomaly"]
    .value_counts()
)

print("\nPERCENTAGE")

print(
    anomaly_df["is_anomaly"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
# INSPECT STRONGEST ANOMALIES

anomaly_records = (
    anomaly_df[
        anomaly_df["is_anomaly"] == 1
    ]
    .sort_values("anomaly_score")
)

print("TOTAL ANOMALIES:", len(anomaly_records))

print("\nSTRONGEST ANOMALIES")
print(
    anomaly_records[
        [
            date_col,
            store_col,
            product_col,
            sales_col,
            inventory_col,
            ordered_col,
            "sales_rolling_7",
            "sales_trend",
            "inventory_to_recent_sales",
            "Price",
            "Discount",
            "Holiday/Promotion",
            "anomaly_score"
        ]
    ].head(20).to_string(index=False)
)

# ANOMALY VS NORMAL COMPARISON
comparison = (
    anomaly_df
    .groupby("is_anomaly")[
        [
            sales_col,
            inventory_col,
            ordered_col,
            "sales_rolling_7",
            "sales_trend",
            "inventory_to_recent_sales",
            "Price",
            "Discount"
        ]
    ]
    .mean()
)

print(comparison)

ANOMALY FEATURES
----------------
Units Sold
Inventory Level
Units Ordered
sales_lag_1
sales_rolling_7
sales_rolling_14
sales_trend
inventory_to_recent_sales
Price
Discount
Holiday/Promotion
Rows: 71000
Features: 11
Isolation Forest trained successfully.
ANOMALY RESULTS
is_anomaly
0    67450
1     3550
Name: count, dtype: int64

PERCENTAGE
is_anomaly
0    95.0
1     5.0
Name: proportion, dtype: float64
TOTAL ANOMALIES: 3550

STRONGEST ANOMALIES
      Date Store ID Product ID  Units Sold  Inventory Level  Units Ordered  sales_rolling_7  sales_trend  inventory_to_recent_sales  Price  Discount  Holiday/Promotion  anomaly_score
2023-06-11     S004      P0009         433              472             27       258.000000     7.500000                   1.822394  45.32        20                  0      -0.097525
2023-02-19     S001      P0020          68               78            187       295.142857    78.642857                   0.263386  12.69         5                  1      -0.097429
20

In [ ]:
# ==========================================
# ANOMALY REASON ANALYSIS
# ==========================================

anomaly_df["anomaly_reason"] = ""

for idx, row in anomaly_df.iterrows():

    if row["is_anomaly"] != 1:
        continue

    reasons = []

    # Sales spike
    if (
        row[sales_col]
        > row["sales_rolling_7"] * 1.5
    ):
        reasons.append("Sales spike")

    # Sales drop
    elif (
        row[sales_col]
        < row["sales_rolling_7"] * 0.5
    ):
        reasons.append("Sales drop")

    # Sudden trend change
    if abs(row["sales_trend"]) > (
        row["sales_rolling_14"] * 0.5
    ):
        reasons.append("Sudden sales trend change")

    # Low inventory coverage
    if row["inventory_to_recent_sales"] < 0.5:
        reasons.append("Low inventory coverage")

    # Very high inventory relative to demand
    elif row["inventory_to_recent_sales"] > 5:
        reasons.append("Inventory unusually high relative to recent sales")

    if reasons:
        anomaly_df.loc[idx, "anomaly_reason"] = (
            ", ".join(reasons)
        )
    else:
        anomaly_df.loc[idx, "anomaly_reason"] = (
            "Unusual multi-feature pattern"
        )


print("ANOMALY REASONS GENERATED")

print(
    anomaly_df[
        anomaly_df["is_anomaly"] == 1
    ][
        [
            date_col,
            store_col,
            product_col,
            sales_col,
            "sales_rolling_7",
            "sales_trend",
            "inventory_to_recent_sales",
            "anomaly_score",
            "anomaly_reason"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ANOMALY REASON SUMMARY
reason_counts = (
    anomaly_df[
        anomaly_df["is_anomaly"] == 1
    ]["anomaly_reason"]
    .value_counts()
)

print("ANOMALY REASON SUMMARY")

print(reason_counts)

ANOMALY REASONS GENERATED
      Date Store ID Product ID  Units Sold  sales_rolling_7  sales_trend  inventory_to_recent_sales  anomaly_score                                                            anomaly_reason
2022-03-27     S001      P0001         289        43.285714   -49.857143                   7.316129      -0.015388 Sales spike, Sudden sales trend change, High inventory relative to demand
2022-04-29     S001      P0001          27       250.000000    66.000000                   1.270916      -0.008174                                                                Sales drop
2022-05-12     S001      P0001          10        47.285714   -23.214286                   8.822485      -0.069668                             Sales drop, High inventory relative to demand
2022-05-13     S001      P0001         357        41.714286    -5.642857                  11.612040      -0.081287                            Sales spike, High inventory relative to demand
2022-05-14     S001      P000

In [ ]:
def analyze_root_causes(
    anomaly_df,
    anomaly_index
):
    row = anomaly_df.loc[anomaly_index]

    product = row[product_col]
    date = row[date_col]

    product_history = anomaly_df[
        anomaly_df[product_col] == product
    ].copy()

    findings = []
    context = []

    # 1. OBSERVED ANOMALY

    current_sales = row[sales_col]
    recent_average = row["sales_rolling_7"]

    if recent_average > 0:
        sales_change = (
            (current_sales - recent_average)
            / recent_average
        ) * 100

        if sales_change > 0:
            anomaly_observation = (
                f"Sales were {sales_change:.1f}% "
                f"above the recent 7-day average"
            )
        else:
            anomaly_observation = (
                f"Sales were {abs(sales_change):.1f}% "
                f"below the recent 7-day average"
            )
    else:
        anomaly_observation = (
            "Recent sales baseline was unavailable"
        )

    # 2. HISTORICAL BASELINE

    if "discount" in detected_columns:

        discount_col = detected_columns["discount"]

        avg_discount = product_history[
            discount_col
        ].mean()

        current_discount = row[discount_col]

        if avg_discount > 0:

            discount_change = (
                (current_discount - avg_discount)
                / avg_discount
            ) * 100

            if abs(discount_change) >= 25:

                direction = (
                    "higher"
                    if discount_change > 0
                    else "lower"
                )

                findings.append(
                    f"Discount was {abs(discount_change):.1f}% "
                    f"{direction} than the product average"
                )

    if "price" in detected_columns:

        price_col = detected_columns["price"]

        avg_price = product_history[
            price_col
        ].mean()

        current_price = row[price_col]

        if avg_price > 0:

            price_change = (
                (current_price - avg_price)
                / avg_price
            ) * 100

            if abs(price_change) >= 10:

                direction = (
                    "higher"
                    if price_change > 0
                    else "lower"
                )

                findings.append(
                    f"Price was {abs(price_change):.1f}% "
                    f"{direction} than the product average"
                )

    # 3. COMPETITOR PRICE CONTEXT
    if (
        "competitor_price" in detected_columns
        and "price" in detected_columns
    ):

        competitor_col = detected_columns[
            "competitor_price"
        ]

        price_col = detected_columns["price"]

        # Historical competitor baseline
        avg_competitor_price = product_history[
            competitor_col
        ].mean()

        current_competitor_price = row[
            competitor_col
        ]

        if avg_competitor_price > 0:

            competitor_change = (
                (
                    current_competitor_price
                    - avg_competitor_price
                )
                / avg_competitor_price
            ) * 100

            if abs(competitor_change) >= 10:

                direction = (
                    "higher"
                    if competitor_change > 0
                    else "lower"
                )

                findings.append(
                    f"Competitor price was "
                    f"{abs(competitor_change):.1f}% "
                    f"{direction} than its historical product  average"
                )

        # Same-day market comparison
        own_price = row[price_col]
        competitor_price = row[competitor_col]

        if competitor_price > 0:

            price_gap = (
                (own_price - competitor_price)
                / competitor_price
            ) * 100

            if abs(price_gap) >= 5:

                direction = (
                    "higher"
                    if price_gap > 0
                    else "lower"
                )

                context.append(
                    f"Own price was {abs(price_gap):.1f}% "
                    f"{direction} than competitor price "
                    f"on the anomaly date"
                )


    # 4. BUSINESS CONTEXT
    if "promotion" in detected_columns:

        promotion_col = detected_columns[
            "promotion"
        ]

        if row[promotion_col] == 1:
            context.append(
                "Promotion was active"
            )

    if "weather" in detected_columns:

        weather_col = detected_columns["weather"]

        context.append(
            f"Weather condition: {row[weather_col]}"
        )

    if "seasonality" in detected_columns:

        seasonality_col = detected_columns[
            "seasonality"
        ]

        context.append(
            f"Seasonality: {row[seasonality_col]}"
        )

    if "region" in detected_columns:

        region_col = detected_columns["region"]

        context.append(
            f"Region: {row[region_col]}"
        )

    # FINAL OUTPUT
    return {
        "product": product,
        "date": date,
        "sales": current_sales,
        "recent_7_day_average": recent_average,
        "anomaly_observation": anomaly_observation,
        "potential_factors": findings,
        "context": context,
        "note": (
            "These factors are associated with the anomaly. "
            "The system does not establish causality."
        )
    }
rca_result = analyze_root_causes(
    anomaly_df,
    strongest_anomaly_index
)

print("ANOMALY ANALYSIS")

print(
    f"Product: {rca_result['product']}"
)

print(
    f"Date: {rca_result['date']}"
)

print()

print("Observed anomaly:")
print(
    f"• Sales: {rca_result['sales']:.0f}"
)

print(
    f"• Recent 7-day average: "
    f"{rca_result['recent_7_day_average']:.0f}"
)

print(
    f"• {rca_result['anomaly_observation']}"
)

print()

print("Potential contributing factors:")

if rca_result["potential_factors"]:

    for factor in rca_result["potential_factors"]:
        print("•", factor)

else:
    print("• No strong historical deviations detected")

print()

print("Business context:")

if rca_result["context"]:

    for item in rca_result["context"]:
        print("•", item)

else:
    print("• No additional contextual factors detected")

print()

print("Note:")
print(rca_result["note"])

ANOMALY ANALYSIS
Product: P0009
Date: 2023-06-11 00:00:00

Observed anomaly:
• Sales: 433
• Recent 7-day average: 258
• Sales were 67.8% above the recent 7-day average

Potential contributing factors:
• Discount was 95.8% higher than the product average
• Price was 17.6% lower than the product average
• Competitor price was 14.8% lower than its historical product  average

Business context:
• Weather condition: Rainy
• Seasonality: Summer
• Region: North

Note:
These factors are associated with the anomaly. The system does not establish causality.


In [ ]:
def generate_decision(
    current_inventory,
    forecasted_demand,
    risk_probability
):
    if forecasted_demand <= 0:
        return "HOLD"

    coverage_ratio = (
        current_inventory / forecasted_demand
    )

    if (
        risk_probability >= 0.75
        and coverage_ratio < 0.20
    ):
        return "URGENT REORDER"

    elif (
        risk_probability >= 0.50
        and coverage_ratio < 0.30
    ):
        return "REORDER"

    elif risk_probability >= 0.50:
        return "MONITOR"

    return "HOLD"


def explain_decision(
    current_inventory,
    forecasted_demand,
    risk_probability,
    decision
):
    coverage_ratio = (
        current_inventory / forecasted_demand
        if forecasted_demand > 0
        else float("inf")
    )

    return (
        f"Current inventory is "
        f"{current_inventory:.0f} units against "
        f"forecasted demand of "
        f"{forecasted_demand:.0f} units. "
        f"Inventory coverage is "
        f"{coverage_ratio:.2f}x and predicted "
        f"stockout risk is "
        f"{risk_probability:.1%}. "
        f"Recommended action: {decision}."
    )


current_inventory = 109
forecasted_demand = 956.002
risk_probability = 0.9745

decision = generate_decision(
    current_inventory,
    forecasted_demand,
    risk_probability
)

explanation = explain_decision(
    current_inventory,
    forecasted_demand,
    risk_probability,
    decision
)

print("DECISION ENGINE")
print("Decision:", decision)
print()
print("Explanation:")
print(explanation)

DECISION ENGINE
Decision: URGENT REORDER

Explanation:
Current inventory is 109 units against forecasted demand of 956 units. Inventory coverage is 0.11x and predicted stockout risk is 97.5%. Recommended action: URGENT REORDER.


In [ ]:
def generate_inventory_decision(
    current_inventory,
    forecasted_demand,
    risk_probability,
    anomaly_reason=None,
    root_causes=None
):
    if forecasted_demand <= 0:
        decision = "HOLD"
    else:
        coverage_ratio = (
            current_inventory / forecasted_demand
        )

        if (
            risk_probability >= 0.75
            and coverage_ratio < 0.20
        ):
            decision = "URGENT REORDER"
        elif (
            risk_probability >= 0.50
            and coverage_ratio < 0.30
        ):
            decision = "REORDER"
        elif risk_probability >= 0.50:
            decision = "MONITOR"
        else:
            decision = "HOLD"

    coverage_ratio = (
        current_inventory / forecasted_demand
        if forecasted_demand > 0
        else float("inf")
    )

    evidence = [
        f"Current inventory: {current_inventory:.0f} units",
        f"Forecasted demand: {forecasted_demand:.0f} units",
        f"Inventory coverage: {coverage_ratio:.2f}x",
        f"Predicted stockout risk: {risk_probability:.1%}"
    ]

    if anomaly_reason:
        evidence.append(
            f"Anomaly signal: {anomaly_reason}"
        )

    if root_causes:
        evidence.extend(
            root_causes
        )

    return {
        "decision": decision,
        "coverage_ratio": coverage_ratio,
        "risk_probability": risk_probability,
        "evidence": evidence
    }


result = generate_inventory_decision(
    current_inventory=109,
    forecasted_demand=956.002,
    risk_probability=0.9745
)

print("INVENTORY DECISION")
print("Decision:", result["decision"])
print()
print("Evidence:")

for item in result["evidence"]:
    print("•", item)

INVENTORY DECISION
Decision: URGENT REORDER

Evidence:
• Current inventory: 109 units
• Forecasted demand: 956 units
• Inventory coverage: 0.11x
• Predicted stockout risk: 97.5%


In [ ]:
anomaly_row = anomaly_df.loc[strongest_anomaly_index]

anomaly_reason = anomaly_row["anomaly_reason"]

rca_result = analyze_root_causes(
    anomaly_df,
    strongest_anomaly_index
)

result = generate_inventory_decision(
    current_inventory=109,
    forecasted_demand=956.002,
    risk_probability=0.9745,
    anomaly_reason=anomaly_reason,
    root_causes=rca_result["potential_factors"]
)

print("INVENTORY DECISION")
print("Decision:", result["decision"])
print()
print("Evidence:")

for item in result["evidence"]:
    print("•", item)

INVENTORY DECISION
Decision: URGENT REORDER

Evidence:
• Current inventory: 109 units
• Forecasted demand: 956 units
• Inventory coverage: 0.11x
• Predicted stockout risk: 97.5%
• Anomaly signal: Sales spike
• Discount was 95.8% higher than the product average
• Price was 17.6% lower than the product average
• Competitor price was 14.8% lower than its historical product  average
